In [ ]:
# Import all necessary packages
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import mean_absolute_error
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from plottable import ColumnDefinition, Table
import rootutils

# Set directories and load data
path_root = str(rootutils.find_root(indicator=".project-root"))
path_plots = f"{path_root}/plots"
path_data = f"{path_root}/data/age-regression"
df_feats = pd.read_excel(f"{path_data}/features.xlsx", index_col=0)
imms = df_feats.index.to_list()
df = pd.read_excel(f"{path_data}/data.xlsx")

# Plot Supplementary Figure S5
df_ctrl = df.loc[df['Status'] == 'Control', :]
df_clocks = pd.read_excel(f"{path_data}/clocks_meta.xlsx", index_col=0)
epiages = df_clocks.index[df_clocks['Type'] == 'Age'].to_list()
epiages.remove('EpInflammAge')

sns.set_theme(style='ticks')
fig = plt.figure(
    figsize=(13, 25),
    layout="constrained"
)
subfigs = fig.subfigures(
    nrows=7,
    ncols=4,
)
for epiage_id, epiage in tqdm(enumerate(epiages)):
    row_id, col_id = divmod(epiage_id, 4)

    axs = subfigs[row_id, col_id].subplot_mosaic(
        [
            ['1'],
            ['2'],
        ],
        height_ratios=[1, 4],
        gridspec_kw={
            "bottom": 0.14,
            "top": 0.95,
            "wspace": 0.33,
            "hspace": 0.01,
        },
    )
    
    ds_table = pd.DataFrame(index=['MAE', "Pearson's R", "Bias"], columns=[epiage])
    mae = mean_absolute_error(df_ctrl['Age'].values, df_ctrl[epiage].values)
    rho, _ = stats.pearsonr(df_ctrl['Age'].values, df_ctrl[epiage].values)
    bias = np.mean(df_ctrl[epiage] - df_ctrl['Age'])
    ds_table.at['MAE', epiage] = f"{mae:0.3f}"
    ds_table.at["Pearson's R", epiage] = f"{rho:0.3f}"
    ds_table.at["Bias", epiage] = f"{bias:0.3f}"
    table_title = f"{epiage}\:({df_clocks.at[epiage, 'Year']})"
    col_defs = [
        ColumnDefinition(
            name="index",
            title=fr"$\mathbf{{{table_title}}}$",
            textprops={"ha": "left"},
            width=4.5,
        ),
        ColumnDefinition(
            name=epiage,
            title='',
            textprops={"ha": "center"},
            width=2.0,
        ),
    ]
    table = Table(
        ds_table,
        column_definitions=col_defs,
        row_dividers=True,
        footer_divider=False,
        ax=axs['1'],
        textprops={"fontsize": 7},
        row_divider_kw={"linewidth": 1, "linestyle": (0, (1, 1))},
        col_label_divider_kw={"linewidth": 1, "linestyle": "-"},
        column_border_kw={"linewidth": 1, "linestyle": "-"},
    ).autoset_fontcolors(colnames=[epiage])
    
    xy_min = df_ctrl[['Age', epiage]].min().min()
    xy_max = df_ctrl[['Age', epiage]].max().max()
    xy_ptp = xy_max - xy_min
    bisect = sns.lineplot(
        x=[xy_min - 0.1 * xy_ptp, xy_max + 0.1 * xy_ptp],
        y=[xy_min - 0.1 * xy_ptp, xy_max + 0.1 * xy_ptp],
        linestyle='--',
        color='black',
        linewidth=1.0,
        zorder=0,
        ax=axs['2']
    )
    regplot = sns.regplot(
        data=df_ctrl,
        x='Age',
        y=epiage,
        color='crimson',
        scatter=False,
        truncate=False,
        ax=axs['2'],
    )
    kdeplot = sns.kdeplot(
        data=df_ctrl,
        x='Age',
        y=epiage,
        fill=True,
        cbar=False,
        color='gray',
        thresh=0.002,
        cut=0,
        legend=False,
        zorder=0,
        ax=axs['2']
    )
    axs['2'].set_xlim(xy_min - 0.1 * xy_ptp, xy_max + 0.1 * xy_ptp)
    axs['2'].set_ylim(xy_min - 0.1 * xy_ptp, xy_max + 0.1 * xy_ptp)
    
fig.savefig(f"{path_plots}/supplementary-figure-s5.png", bbox_inches='tight', dpi=200)
fig.savefig(f"{path_plots}/supplementary-figure-s5.pdf", bbox_inches='tight')
plt.close(fig)